# CatBoost: обучение отдельно по каждому resist_id

Обучение модели CatBoost **отдельно по каждому resist_id** с перебором комбинаций гиперпараметров. Для каждого resist_id выбирается набор параметров с наилучшим показателем **Качество** (гармоническое среднее Точности и Стабильности). Лучшие параметры сохраняются в датафрейм.

## 1. Конфигурация

Регулируемые параметры:
- **RESIST_IDS_TO_TRAIN** — список resist_id для обучения
- **CATBOOST_PARAM_GRID** — диапазоны гиперпараметров для перебора

In [ ]:
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

# --- Список resist_id для обучения (можно менять) ---
# Конкретный список или None = все доступные resist_id из данных
RESIST_IDS_TO_TRAIN = [31, 47, 49, 59, 84, 88, 128, 159, 161, 195]  # или None

# --- Диапазоны параметров CatBoost (регулировать при необходимости) ---
CATBOOST_PARAM_GRID = {
    "iterations": [500, 1000, 1500],
    "depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "loss_function": ["RMSE", "MAE"],
    "early_stopping_rounds": [30, 50, 100],
    "l2_leaf_reg": [1, 10],
    "min_data_in_leaf": [1, 10],
    "random_strength": [0, 0.5],
    "bagging_temperature": [0, 0.5],
    "border_count": [32, 128],
}

# Фиксированные параметры (не перебираются)
CATBOOST_FIXED = {
    "random_seed": 42,
    "verbose": 0,
}

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
FILTER_YEAR = 2025

## 2. Загрузка и подготовка данных (на основе build_model39_dataframe_v2)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def get_calc_value(calc_obj: dict, key: str):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val


# Загрузка CSV
dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)
mask_ok = ~df_all["cancel_approve"] & ~mask_predict_record & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    workday = get_calc_value(calc_obj, "workDayMonth_2")
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        records.append({
            "resist_id": row.get("resist_id"),
            "division_title": parse_division_title(row.get("division_name")),
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": workday,
            "calculate_minute": payload.get("calculate_minute"),
        })

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")
df_model39["coefficient_fulltime_staff"] = df_model39["calculate_minute"] / 60 / 148.3
df_model39["date"] = pd.to_datetime(dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce").dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# Фильтрация нулевых месяцев 2025
df_model39["_grp"] = df_model39["resist_id"].astype(str) + "|" + df_model39["division_title"].astype(str) + "|" + df_model39["operation_id"].astype(str)
yr_mask = df_model39["year"] == FILTER_YEAR
drop_idx = set()
for g in df_model39["_grp"].unique():
    grp = df_model39[(df_model39["_grp"] == g) & yr_mask].sort_values("month")
    for idx in grp.index:
        fv, fp = df_model39.at[idx, "fact_value"], df_model39.at[idx, "fact_predict"]
        if (pd.isna(fv) or float(fv) == 0) and (pd.isna(fp) or float(fp) == 0):
            drop_idx.add(idx)
        else:
            break
df_model39 = df_model39.drop(index=list(drop_idx)).drop(columns=["_grp"]).reset_index(drop=True)

# Feature Engineering
group = df_model39.groupby("operation_id")
df_model39["lag_1"] = group["fact_value"].shift(1)
df_model39["lag_12"] = group["fact_value"].shift(12)
df_model39["roll_mean_3"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
df_model39["roll_mean_6"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(6, min_periods=6).mean())
df_model39["trend"] = (df_model39["lag_1"] / df_model39["roll_mean_3"]).replace([np.inf, -np.inf], 0)

num_cols_clean = df_model39.select_dtypes(include=["number"]).columns
df_model39[num_cols_clean] = df_model39[num_cols_clean].fillna(0)
df_model39 = df_model39.infer_objects(copy=False)

print(f"Итоговых строк: {len(df_model39)}")
print(f"Уникальных resist_id: {df_model39['resist_id'].nunique()}")

## 3. Функции: метрика Качество и обучение

In [ ]:
def calc_accuracy_pct(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Точность по строке: 100 - MAPE%, с ограничением [0, 100]."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0, 0, 100)
    return acc


def calc_quality(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Качество = 2 * Точность * Стабильность / (Точность + Стабильность)."""
    acc = calc_accuracy_pct(y_true, y_pred)
    mean_acc = float(np.mean(acc))
    if mean_acc == 0:
        return 0.0
    cv = np.std(acc, ddof=0) / mean_acc
    stab = max(0.0, (1 - cv)) * 100
    if mean_acc + stab == 0:
        return 0.0
    return 2 * mean_acc * stab / (mean_acc + stab)


# Признаки для обучения (без resist_id — при обучении по одному resist_id он константен)
CAT_COLS = ["operation_id", "division_title"]
NUM_COLS = [
    "month", "year", "workDayMonth_2", "calculate_minute", "coefficient_fulltime_staff",
    "lag_1", "lag_12", "roll_mean_3", "roll_mean_6", "trend",
]
FEATURE_COLS = CAT_COLS + NUM_COLS
TARGET_COL = "fact_value"


def train_and_evaluate(df_rid: pd.DataFrame, params: dict) -> tuple[float, np.ndarray]:
    """Обучает CatBoost на данных одного resist_id, возвращает (Качество, pred)."""
    last_year = df_rid["year"].max()
    train_df = df_rid[df_rid["year"] < last_year].copy()
    test_df = df_rid[df_rid["year"] == last_year].copy()
    if train_df.empty or test_df.empty:
        return float("-inf"), np.array([])

    for c in CAT_COLS:
        train_df[c] = train_df[c].astype(str).fillna("")
        test_df[c] = test_df[c].astype(str).fillna("")

    X_train = train_df[FEATURE_COLS]
    y_train = train_df[TARGET_COL]
    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COL].values

    cat_idx = [FEATURE_COLS.index(c) for c in CAT_COLS]
    sw = train_df["calculate_minute"].fillna(train_df["calculate_minute"].mean()).astype(float)

    model = CatBoostRegressor(
        **{**CATBOOST_FIXED, **params},
        cat_features=cat_idx,
    )
    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        sample_weight=sw,
    )
    pred = np.maximum(model.predict(X_test), 0)
    quality = calc_quality(y_test, pred)
    return quality, pred

## 4. Цикл: перебор resist_id и grid search параметров

In [ ]:
# Если RESIST_IDS_TO_TRAIN = None, берём все уникальные resist_id из данных
resist_ids_to_train = RESIST_IDS_TO_TRAIN
if resist_ids_to_train is None:
    resist_ids_to_train = sorted(df_model39["resist_id"].dropna().astype(int).unique().tolist())
    print(f"Используются все resist_id из данных: {len(resist_ids_to_train)} шт.")

param_names = list(CATBOOST_PARAM_GRID.keys())
param_values = list(CATBOOST_PARAM_GRID.values())
param_combinations = list(itertools.product(*param_values))
n_combinations = len(param_combinations)
print(f"Всего комбинаций параметров: {n_combinations}")
print(f"resist_id для обучения: {resist_ids_to_train}")
print("---")

In [ ]:
results_rows = []

for rid in resist_ids_to_train:
    df_rid = df_model39[df_model39["resist_id"] == rid].copy()
    if df_rid.empty:
        print(f"resist_id={rid}: нет данных, пропуск")
        continue
    n_train = len(df_rid[df_rid["year"] < df_rid["year"].max()])
    n_test = len(df_rid[df_rid["year"] == df_rid["year"].max()])
    if n_train < 10 or n_test < 1:
        print(f"resist_id={rid}: недостаточно данных (train={n_train}, test={n_test}), пропуск")
        continue

    best_quality = float("-inf")
    best_params = None
    best_combo_idx = -1

    for combo_idx, combo in enumerate(param_combinations):
        params = dict(zip(param_names, combo))
        quality, _ = train_and_evaluate(df_rid, params)
        if quality > best_quality:
            best_quality = quality
            best_params = params.copy()
            best_combo_idx = combo_idx

    if best_params is not None:
        row = {
            "resist_id": rid,
            "Качество": round(best_quality, 4),
            "n_train": n_train,
            "n_test": n_test,
        }
        for k, v in best_params.items():
            row[f"param_{k}"] = v
        results_rows.append(row)
        print(f"resist_id={rid}: Качество={best_quality:.2f} | params: {best_params}")

df_best_params = pd.DataFrame(results_rows)
print("\n--- Итоговый датафрейм с лучшими параметрами по каждому resist_id ---")
df_best_params

## 5. Сохранение результатов

In [ ]:
OUT_PATH = Path("../data/processed/catboost_best_params_per_resist_id.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_best_params.to_csv(OUT_PATH, index=False)
print(f"Сохранено: {OUT_PATH}")